# Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import re

# Get data

In [2]:
combats = pd.read_csv("combats.csv")
pokemon = pd.read_csv("pokemon.csv", index_col=0)  # use '#' as index

# Data handling

In [3]:
importances_top_n = 15

# ensure indices/dtypes
pokemon.index = pokemon.index.astype(int)
combats[['First_pokemon','Second_pokemon','Winner']] = combats[['First_pokemon','Second_pokemon','Winner']].astype(int)
if pokemon['Legendary'].dtype == bool or pokemon['Legendary'].dtype == object:
    pokemon['Legendary'] = pokemon['Legendary'].astype(int)

# prefix and merge stats for both fighters
p1 = pokemon.add_prefix('p1_')
p2 = pokemon.add_prefix('p2_')
df = combats.merge(p1, left_on='First_pokemon', right_index=True, how='left')
df = df.merge(p2, left_on='Second_pokemon', right_index=True, how='left')

# label: 1 if first pokemon is the winner, else 0
df['first_win'] = (df['Winner'] == df['First_pokemon']).astype(int)

# gather all types present in the pokemon table
all_types = pd.unique(pokemon[['Type 1', 'Type 2']].values.ravel())
all_types = [t for t in all_types if pd.notna(t)]

# create multi-hot columns for p1 and p2
for t in all_types:
    safe = re.sub(r'[^0-9a-zA-Z]+', '_', str(t))
    df[f'p1_type_{safe}'] = ((df['p1_Type 1'] == t) | (df['p1_Type 2'] == t)).astype(int)
    df[f'p2_type_{safe}'] = ((df['p2_Type 1'] == t) | (df['p2_Type 2'] == t)).astype(int)

# select numeric feature columns from the prefixed stats (drop name/type columns)
feat_cols = [c for c in df.columns if (c.startswith('p1_') or c.startswith('p2_')) and np.issubdtype(df[c].dtype, np.number)]
X = df[feat_cols].fillna(0)  # handle any missing joins conservatively
y = df['first_win']

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Features: ")
print(X.columns.tolist())

Features: 
['p1_HP', 'p1_Attack', 'p1_Defense', 'p1_Sp. Atk', 'p1_Sp. Def', 'p1_Speed', 'p1_Generation', 'p1_Legendary', 'p2_HP', 'p2_Attack', 'p2_Defense', 'p2_Sp. Atk', 'p2_Sp. Def', 'p2_Speed', 'p2_Generation', 'p2_Legendary', 'p1_type_Grass', 'p2_type_Grass', 'p1_type_Poison', 'p2_type_Poison', 'p1_type_Fire', 'p2_type_Fire', 'p1_type_Flying', 'p2_type_Flying', 'p1_type_Dragon', 'p2_type_Dragon', 'p1_type_Water', 'p2_type_Water', 'p1_type_Bug', 'p2_type_Bug', 'p1_type_Normal', 'p2_type_Normal', 'p1_type_Electric', 'p2_type_Electric', 'p1_type_Ground', 'p2_type_Ground', 'p1_type_Fairy', 'p2_type_Fairy', 'p1_type_Fighting', 'p2_type_Fighting', 'p1_type_Psychic', 'p2_type_Psychic', 'p1_type_Rock', 'p2_type_Rock', 'p1_type_Steel', 'p2_type_Steel', 'p1_type_Ice', 'p2_type_Ice', 'p1_type_Ghost', 'p2_type_Ghost', 'p1_type_Dark', 'p2_type_Dark']


# Random forest

In [4]:
from sklearn.model_selection import RandomizedSearchCV

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_param_grid = {
    'n_estimators': [100, 200, 500, 1000],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],  # Number of features per split
    'bootstrap': [True, False]               # Whether to use bootstrap samples
}

rf_random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_grid,
    n_iter=20,           # Number of combinations to try
    scoring='accuracy',  # Or 'neg_log_loss' to match your XGBoost setup
    cv=3, 
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rf_random_search.fit(X_train, y_train)

best_rf = rf_random_search.best_estimator_

print(f"Best RF Parameters: {rf_random_search.best_params_}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best RF Parameters: {'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': 30, 'bootstrap': True}


In [ ]:

rf_base.fit(X_train, y_train)

# Helper functions

In [5]:
# Build once: exact lookup from typed name -> canonical dataset name
_name_lookup = {n.strip(): n for n in pokemon['Name'].dropna().astype(str)}

def _resolve_name(name: str) -> str:
    key = str(name).strip()
    if key in _name_lookup:
        return _name_lookup[key]
    raise ValueError(f"Pokemon '{name}' not found in pokemon.csv")

def _build_feature_row(name1: str, name2: str) -> pd.DataFrame:
    n1 = _resolve_name(name1)
    n2 = _resolve_name(name2)

    r1 = pokemon.loc[pokemon['Name'] == n1].iloc[0]
    r2 = pokemon.loc[pokemon['Name'] == n2].iloc[0]

    # start all features at 0, then fill what applies
    row = {c: 0 for c in feat_cols}

    # numeric stats used by your model
    base_num = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']
    for c in base_num:
        p1c = f"p1_{c}"
        p2c = f"p2_{c}"
        if p1c in row:
            row[p1c] = int(r1[c]) if c == 'Legendary' else float(r1[c])
        if p2c in row:
            row[p2c] = int(r2[c]) if c == 'Legendary' else float(r2[c])

    # multi-hot types
    t1 = {str(r1.get('Type 1', '')).strip(), str(r1.get('Type 2', '')).strip()}
    t2 = {str(r2.get('Type 1', '')).strip(), str(r2.get('Type 2', '')).strip()}

    for c in feat_cols:
        if c.startswith("p1_type_"):
            typ = c.replace("p1_type_", "").replace("_", " ")
            row[c] = int(typ in t1)
        elif c.startswith("p2_type_"):
            typ = c.replace("p2_type_", "").replace("_", " ")
            row[c] = int(typ in t2)

    return pd.DataFrame([row], columns=feat_cols), n1, n2

def predict(name1: str, name2: str):
    """
    predict('Pikachu', 'Charizard')
    Prints who wins based on trained rf model.
    """
    X_new, n1, n2 = _build_feature_row(name1, name2)
    pred = int(best_rf.predict(X_new)[0])  # 1 => first wins, 0 => second wins
    winner = n1 if pred == 1 else n2
    print(f"{n1} vs {n2} -> Winner: {winner}")

# Tests

In [6]:
predict("Pikachu", "Chimchar")

# Evaluate base_rf vs best_rf
from sklearn.metrics import accuracy_score, f1_score

base_rf = rf_base
models = {"base_rf": base_rf, "best_rf": best_rf}

for name, model in models.items():
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    acc_train = accuracy_score(y_train, y_train_pred)
    f1_train = f1_score(y_train, y_train_pred)
    acc_test = accuracy_score(y_test, y_test_pred)
    f1_test = f1_score(y_test, y_test_pred)

    print(f"== {name} ==")
    print(f"Train Accuracy: {acc_train:.4f}")
    print(f"Train F1 Score: {f1_train:.4f}")
    print(f"Test Accuracy:  {acc_test:.4f}")
    print(f"Test F1 Score:  {f1_test:.4f}")
    print()


Pikachu vs Chimchar -> Winner: Pikachu


NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.